# Sequence-to-Sequence Models for Multi-Step Forecasting

In the previous notebook we used **recursive forecasting** to generate
multi-step predictions: feed the model's own output back as input,
one step at a time.  This works, but errors accumulate with each step.

**Sequence-to-sequence (Seq2Seq)** models offer a better approach:
an **encoder** compresses the input sequence into a fixed-length
representation, and a **decoder** generates the entire output sequence
in one pass.

**Contents:**
1. Multi-step forecasting strategies: recursive, direct, seq2seq
2. Encoder-decoder architecture
3. Teacher forcing
4. Data preparation for seq2seq
5. Building an encoder-decoder LSTM with the Keras Functional API
6. Training and evaluation
7. Practical example: predict 12 months from 24 months input
8. Comparison with recursive approach

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras import layers

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

np.random.seed(42)
keras.utils.set_random_seed(42)

print(f"Keras version: {keras.__version__}")

---
## 1. Multi-Step Forecasting Strategies

Given an input window of $w$ past values, we want to predict the next
$h$ future values.  There are three main strategies:

### Recursive (Iterated)

Train a one-step model, then feed its predictions back as input:

$$
\hat{y}_{t+1} = f(y_{t-w+1}, \ldots, y_t), \quad
\hat{y}_{t+2} = f(y_{t-w+2}, \ldots, \hat{y}_{t+1}), \quad \ldots
$$

- **Pro**: only one model to train
- **Con**: errors compound at each step

### Direct (Multi-Output)

Train a separate model (or multi-output model) that directly predicts
all $h$ future values:

$$
[\hat{y}_{t+1}, \ldots, \hat{y}_{t+h}] = f(y_{t-w+1}, \ldots, y_t)
$$

- **Pro**: no error accumulation
- **Con**: ignores temporal dependencies between output steps

### Sequence-to-Sequence (Encoder-Decoder)

Use an encoder to compress the input into a context vector, then
a decoder to generate the output sequence *autoregressively*:

- **Pro**: models temporal dependencies in both input and output
- **Con**: more complex architecture, more data needed

In [ ]:
# Visual comparison of the three strategies
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Common data
t_in = np.arange(6)
t_out = np.arange(6, 9)
y_in = [2, 3, 2.5, 4, 3.5, 5]
y_out = [4.5, 5.5, 5.0]

for ax in axes:
    ax.plot(t_in, y_in, "bo-", label="Input")
    ax.set_ylim(0, 7)
    ax.set_xlabel("Time")

# Recursive
axes[0].set_title("Recursive")
for i, t in enumerate(t_out):
    axes[0].plot(t, y_out[i], "rs", markersize=10)
    if i > 0:
        axes[0].annotate("", xy=(t, y_out[i]), xytext=(t - 1, y_out[i - 1]),
                        arrowprops=dict(arrowstyle="->", color="red", lw=1.5))
axes[0].annotate("", xy=(t_out[0], y_out[0]), xytext=(t_in[-1], y_in[-1]),
                arrowprops=dict(arrowstyle="->", color="red", lw=1.5))

# Direct
axes[1].set_title("Direct (Multi-Output)")
for t, y in zip(t_out, y_out):
    axes[1].plot(t, y, "gs", markersize=10)
    axes[1].annotate("", xy=(t, y), xytext=(t_in[-1], y_in[-1]),
                    arrowprops=dict(arrowstyle="->", color="green", lw=1, alpha=0.5))

# Seq2Seq
axes[2].set_title("Seq2Seq (Encoder-Decoder)")
# Encoder arrow to context
axes[2].annotate("context", xy=(5.5, 3.5), fontsize=9, color="purple",
                ha="center", va="bottom")
axes[2].plot(t_out, y_out, "ms-", markersize=10, label="Decoder output")

plt.tight_layout()
plt.show()

print("Recursive: one-step model, feed predictions back (errors compound).")
print("Direct: predict all steps at once (ignores output dependencies).")
print("Seq2Seq: encoder compresses input, decoder generates output sequence.")

---
## 2. Encoder-Decoder Architecture

The seq2seq model consists of two parts:

### Encoder

An LSTM that processes the input sequence and produces:
- A **context vector** (the final hidden state $h_T$ and cell state $C_T$)
- The output sequence is discarded

$$
h_t^{\text{enc}}, C_t^{\text{enc}} = \text{LSTM}_{\text{enc}}(x_t, h_{t-1}^{\text{enc}}, C_{t-1}^{\text{enc}})
$$

### Decoder

An LSTM initialized with the encoder's final state that generates the
output sequence one step at a time:

$$
h_t^{\text{dec}}, C_t^{\text{dec}} = \text{LSTM}_{\text{dec}}(\hat{y}_{t-1}, h_{t-1}^{\text{dec}}, C_{t-1}^{\text{dec}})
$$
$$
\hat{y}_t = W_o h_t^{\text{dec}} + b_o
$$

The context vector acts as a "summary" of the input sequence, and the
decoder uses it to produce the output.

---
## 3. Teacher Forcing

During training, the decoder can receive input in two ways:

1. **Autoregressive** (no teacher forcing): the decoder's own previous
   prediction $\hat{y}_{t-1}$ is used as input for step $t$

2. **Teacher forcing**: the *true* value $y_{t-1}$ is used as input
   for step $t$, regardless of what the model predicted

**Teacher forcing** speeds up training significantly because the decoder
always sees correct inputs, preventing error propagation during training.
However, at inference time we must use autoregressive decoding (since we
do not have future ground truth), creating a **train-test mismatch**.

A common compromise is **scheduled sampling**: start with teacher forcing
and gradually switch to autoregressive decoding during training.

For simplicity, we will use a direct decoder approach that generates
all output steps in parallel using `RepeatVector`.

---
## 4. Data Preparation

For seq2seq, we create (input_sequence, output_sequence) pairs:
- Input: 24 months of history
- Output: 12 months of future values

In [ ]:
# Load airline passengers data
airline = pd.read_csv(DATA_DIR / "airline_passengers.csv")
airline.columns = ["Month", "Passengers"]
airline["Month"] = pd.to_datetime(airline["Month"])
airline = airline.set_index("Month")

print(f"Shape: {airline.shape}")
print(f"Date range: {airline.index.min().date()} to {airline.index.max().date()}")

In [ ]:
# Scale data
scaler = MinMaxScaler()
values = airline["Passengers"].values.reshape(-1, 1).astype("float32")
scaled = scaler.fit_transform(values).flatten()

print(f"Scaled range: [{scaled.min():.4f}, {scaled.max():.4f}]")

In [ ]:
def create_seq2seq_data(data, input_len, output_len):
    """Create (encoder_input, decoder_target) pairs for seq2seq.

    Parameters
    ----------
    data : array-like, shape (n,)
        The time series values.
    input_len : int
        Length of the encoder input sequence.
    output_len : int
        Length of the decoder output sequence.

    Returns
    -------
    X : ndarray, shape (n_samples, input_len, 1)
    y : ndarray, shape (n_samples, output_len, 1)
    """
    X, y = [], []
    for i in range(len(data) - input_len - output_len + 1):
        X.append(data[i:i + input_len])
        y.append(data[i + input_len:i + input_len + output_len])
    X = np.array(X).reshape(-1, input_len, 1)
    y = np.array(y).reshape(-1, output_len, 1)
    return X, y

In [ ]:
# Create seq2seq pairs: 24 months in, 12 months out
INPUT_LEN = 24
OUTPUT_LEN = 12

X_s2s, y_s2s = create_seq2seq_data(scaled, INPUT_LEN, OUTPUT_LEN)

print(f"X shape: {X_s2s.shape}  (samples, input_len, features)")
print(f"y shape: {y_s2s.shape}  (samples, output_len, features)")
print(f"Total samples: {X_s2s.shape[0]}")

In [ ]:
# Train/test split -- hold out the last sample (last 12 months)
# With such a small dataset we only have ~100 samples
n_test_samples = 12  # last 12 sliding-window samples

X_train, X_test = X_s2s[:-n_test_samples], X_s2s[-n_test_samples:]
y_train, y_test = y_s2s[:-n_test_samples], y_s2s[-n_test_samples:]

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

---
## 5. Building the Encoder-Decoder with the Functional API

We use the **Keras Functional API** for more control over the
architecture.  The model has three parts:

1. **Encoder**: LSTM that reads the input and produces a context vector
2. **RepeatVector**: repeats the context vector for each output time step
3. **Decoder**: LSTM that processes the repeated context and generates
   the output sequence via a TimeDistributed Dense layer

In [ ]:
# Encoder-Decoder using Sequential API with RepeatVector
model_s2s = keras.Sequential([
    # Encoder
    layers.LSTM(64, input_shape=(INPUT_LEN, 1)),

    # Bridge: repeat context vector for each output step
    layers.RepeatVector(OUTPUT_LEN),

    # Decoder
    layers.LSTM(64, return_sequences=True),

    # Output: one prediction per time step
    layers.TimeDistributed(layers.Dense(1)),
])

model_s2s.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_s2s.summary()

In [ ]:
# Now let us also build this using the Functional API for more flexibility
# This gives us explicit access to the encoder states

# Encoder
encoder_input = keras.Input(shape=(INPUT_LEN, 1), name="encoder_input")
encoder_lstm = layers.LSTM(64, return_state=True, name="encoder_lstm")
encoder_output, state_h, state_c = encoder_lstm(encoder_input)

# Decoder
# Use RepeatVector to create decoder input from context
decoder_input = layers.RepeatVector(OUTPUT_LEN)(encoder_output)
decoder_lstm = layers.LSTM(64, return_sequences=True, name="decoder_lstm")
decoder_output = decoder_lstm(decoder_input, initial_state=[state_h, state_c])

# Output layer
output = layers.TimeDistributed(layers.Dense(1), name="output")(decoder_output)

model_func = keras.Model(inputs=encoder_input, outputs=output, name="seq2seq")
model_func.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_func.summary()

---
## 6. Training and Evaluation

In [ ]:
# Train the functional API model with early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1,
)

history = model_func.fit(
    X_train, y_train,
    epochs=200,
    batch_size=8,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=0,
)

print(f"Training stopped at epoch {len(history.history['loss'])}")
print(f"Best val loss: {min(history.history['val_loss']):.6f}")

In [ ]:
# Learning curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history["loss"], label="Train")
ax.plot(history.history["val_loss"], label="Validation")
best_epoch = np.argmin(history.history["val_loss"])
ax.axvline(best_epoch, color="red", linestyle="--", alpha=0.6,
           label=f"Best epoch ({best_epoch})")
ax.set_title("Seq2Seq Learning Curves")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Practical Example: 12-Month Forecast from 24 Months

We evaluate the model by using the last available 24-month window
to predict the final 12 months of the dataset.

In [ ]:
# Use the very last test sample: 24 months -> 12 months
X_final = X_test[-1:]
y_final_scaled = y_test[-1].flatten()

# Predict
pred_scaled = model_func.predict(X_final, verbose=0).flatten()

# Inverse transform
pred_orig = scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
actual_orig = scaler.inverse_transform(y_final_scaled.reshape(-1, 1)).flatten()

mae_s2s = mean_absolute_error(actual_orig, pred_orig)
rmse_s2s = np.sqrt(mean_squared_error(actual_orig, pred_orig))

print(f"Seq2Seq 12-Month Forecast")
print(f"  MAE:  {mae_s2s:.2f}")
print(f"  RMSE: {rmse_s2s:.2f}")

In [ ]:
# Visualize the 12-month forecast
forecast_dates = airline.index[-OUTPUT_LEN:]
input_dates = airline.index[-(INPUT_LEN + OUTPUT_LEN):-OUTPUT_LEN]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(airline.index, airline["Passengers"], color="tab:blue", alpha=0.4, label="Full series")
ax.plot(input_dates, airline.loc[input_dates, "Passengers"],
        color="tab:blue", linewidth=2, label="Encoder input (24 months)")
ax.plot(forecast_dates, actual_orig, color="tab:green", marker="o",
        markersize=5, label="Actual (12 months)")
ax.plot(forecast_dates, pred_orig, color="tab:red", linestyle="--",
        marker="s", markersize=5, label=f"Seq2Seq forecast (MAE={mae_s2s:.1f})")
ax.axvline(forecast_dates[0], color="grey", linestyle=":", alpha=0.6)
ax.set_title("Seq2Seq: 24 Months In -> 12 Months Out")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Comparison: Seq2Seq vs Recursive vs Direct

Let us compare all three multi-step strategies on the same forecast.

In [ ]:
# Direct model: LSTM encoder -> Dense output (all 12 steps at once)
model_direct = keras.Sequential([
    layers.LSTM(64, input_shape=(INPUT_LEN, 1)),
    layers.Dense(OUTPUT_LEN),  # directly predict all 12 steps
])

model_direct.compile(optimizer="adam", loss="mse")

# Need y reshaped for direct model: (samples, output_len)
y_train_direct = y_train.reshape(-1, OUTPUT_LEN)
y_test_direct = y_test.reshape(-1, OUTPUT_LEN)

model_direct.fit(
    X_train, y_train_direct,
    epochs=200, batch_size=8, validation_split=0.15,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=20, restore_best_weights=True)],
    verbose=0,
)

pred_direct_s = model_direct.predict(X_final, verbose=0).flatten()
pred_direct = scaler.inverse_transform(pred_direct_s.reshape(-1, 1)).flatten()

mae_direct = mean_absolute_error(actual_orig, pred_direct)
print(f"Direct model MAE: {mae_direct:.2f}")

In [ ]:
# Recursive model: one-step LSTM, fed back recursively
model_recur = keras.Sequential([
    layers.LSTM(64, input_shape=(INPUT_LEN, 1)),
    layers.Dense(1),
])

# Create one-step training data
def create_sequences_1step(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X).reshape(-1, window_size, 1), np.array(y)

X_1step, y_1step = create_sequences_1step(scaled, INPUT_LEN)
X_1_train = X_1step[:-OUTPUT_LEN]
y_1_train = y_1step[:-OUTPUT_LEN]

model_recur.compile(optimizer="adam", loss="mse")
model_recur.fit(
    X_1_train, y_1_train,
    epochs=200, batch_size=8, validation_split=0.15,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=20, restore_best_weights=True)],
    verbose=0,
)

# Recursive prediction
window = scaled[-(INPUT_LEN + OUTPUT_LEN):-OUTPUT_LEN].copy()
recursive_preds_s = []
for _ in range(OUTPUT_LEN):
    pred = model_recur.predict(window.reshape(1, -1, 1), verbose=0)[0, 0]
    recursive_preds_s.append(pred)
    window = np.append(window[1:], pred)

pred_recursive = scaler.inverse_transform(
    np.array(recursive_preds_s).reshape(-1, 1)).flatten()
mae_recursive = mean_absolute_error(actual_orig, pred_recursive)
print(f"Recursive model MAE: {mae_recursive:.2f}")

In [ ]:
# Comparison visualization
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(forecast_dates, actual_orig, color="black", marker="o",
        markersize=6, linewidth=2, label="Actual", zorder=5)
ax.plot(forecast_dates, pred_orig, linestyle="--", marker="s",
        markersize=5, label=f"Seq2Seq (MAE={mae_s2s:.1f})")
ax.plot(forecast_dates, pred_direct, linestyle="-.", marker="^",
        markersize=5, label=f"Direct (MAE={mae_direct:.1f})")
ax.plot(forecast_dates, pred_recursive, linestyle=":", marker="D",
        markersize=5, label=f"Recursive (MAE={mae_recursive:.1f})")

ax.set_title("Multi-Step Forecasting: 12 Months Ahead")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
comparison = pd.DataFrame([
    {"Strategy": "Recursive", "MAE": mae_recursive,
     "Pros": "Simple, one model", "Cons": "Error accumulation"},
    {"Strategy": "Direct", "MAE": mae_direct,
     "Pros": "No error accumulation", "Cons": "Ignores output dependencies"},
    {"Strategy": "Seq2Seq", "MAE": mae_s2s,
     "Pros": "Models output dependencies", "Cons": "More complex, needs more data"},
])

print(comparison.to_string(index=False))

---
## Summary

- **Three strategies** for multi-step forecasting:
  - *Recursive*: one-step model with feedback (errors accumulate)
  - *Direct*: predict all steps at once (ignores output temporal structure)
  - *Seq2Seq*: encoder-decoder architecture (best of both worlds)

- The **encoder** compresses the input sequence into a context vector
  $(h_T, C_T)$; the **decoder** generates the output sequence from it

- **Teacher forcing** uses true values as decoder input during training
  (faster convergence, but train-test mismatch)

- The **Keras Functional API** gives explicit control over encoder states
  and decoder initialization

- With small datasets (like airline passengers with 144 points), seq2seq
  models may not have enough data to outperform simpler approaches.
  They shine on **larger datasets** with complex patterns

- Next notebook: modern architectures (TCN, N-BEATS, TFT, foundation
  models) that push the boundaries of deep learning for time series

In [ ]:
print("Next notebook: Modern architectures -- TCN, N-BEATS, TFT, and foundation models.")